# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write a Rust `audio_sample.rs` file.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [ ]:
from typing import TYPE_CHECKING
from pathlib import Path
import random
import gc
import os

import numpy as np
import tensorflow as tf

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

tf.config.optimizer.set_jit(False)

# Also, limit GPU memory growth to prevent the 'factory registration' errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
# Disable the oneDNN floating point noise
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Prevent TF from grabbing all VRAM
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
# Verify it's off
print(f"XLA Status: {tf.config.optimizer.get_jit()}")

# Reproducibility
SEED = 3407
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Paths
REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR = REPO_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = REPO_ROOT / "dataset"

OUT_TFLITE = MODELS_DIR / "cnn_time_tf.tflite"
OUT_AUDIO_RS = SRC_DIR / "audio_sample.rs"

# Audio geometry (must match Rust)
SAMPLE_RATE = 16000
FRAME_LENGTH = 1024
FRAME_STEP = 256
TARGET_FRAMES = 184
TARGET_AUDIO_LEN = (TARGET_FRAMES - 1) * FRAME_STEP + FRAME_LENGTH

print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)

In [ ]:
def fix_audio_length(audio):
    # audio: [batch, time, 1] from keras audio_dataset_from_directory
    audio = tf.squeeze(audio, axis=-1)  # [batch, time]
    audio = audio[:, :TARGET_AUDIO_LEN]
    current_len = tf.shape(audio)[1]
    pad_len = tf.maximum(0, TARGET_AUDIO_LEN - current_len)
    audio = tf.pad(audio, [[0, 0], [0, pad_len]])
    audio = tf.ensure_shape(audio, [None, TARGET_AUDIO_LEN])
    audio = tf.expand_dims(audio, axis=-1)  # [batch, time, 1]
    return audio


def to_features(audio, label):
    audio = fix_audio_length(audio)
    return audio, label


train_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "training",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=True,
    seed=3407,
)
val_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "validation",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
test_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
label_names = np.array(train_ds_raw.class_names)
num_labels = len(label_names)
print("Classes:", label_names)
train_ds = train_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()
val_ds = val_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()
test_ds = test_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_AUDIO_LEN, 1)),
    keras.layers.Conv1D(4, 3, activation="relu"),
    keras.layers.MaxPooling1D(pool_size=2, strides=2),
    keras.layers.Conv1D(8, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=2, steps_per_epoch=len(train_ds_raw), validation_steps=len(val_ds_raw))
print("Test metrics:", model.evaluate(test_ds, verbose=0, steps=len(test_ds_raw)))

In [ ]:

val_specs = []
for x_batch, _ in test_ds.unbatch().take(100):
    sample = x_batch.numpy().astype(np.float32)
    sample = np.reshape(sample, (1, TARGET_AUDIO_LEN, 1))
    val_specs.append(sample)

def representative_data_gen():
    for sample in val_specs:
        yield [sample]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

converter._experimental_lower_tensor_list_ops = True

try:
    with tf.device('/CPU:0'):
        tflite_bytes = converter.convert()
    OUT_TFLITE.write_bytes(tflite_bytes)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

In [ ]:
raw_sample_ds = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=1,
    shuffle=False,
)

clips_by_label = {0: [], 1: []}
for audio_batch, label_batch in raw_sample_ds.unbatch():
    label = int(label_batch.numpy())
    if len(clips_by_label[label]) < 2:
        fixed = fix_audio_length(tf.expand_dims(audio_batch, 0))[0].numpy().astype(np.float32)
        clips_by_label[label].append(fixed)
    if len(clips_by_label[0]) == 2 and len(clips_by_label[1]) == 2:
        break

ordered = [
    ("non_target", clips_by_label[0][0]),
    ("target", clips_by_label[1][0]),
    ("non_target", clips_by_label[0][1]),
    ("target", clips_by_label[1][1]),
]

rs = []
rs.append("// Generated by building_tensorflow/cnn_time.ipynb\n")
rs.append(f"pub const SAMPLE_RATE: usize = {SAMPLE_RATE};\n\n")
rs.append("pub struct TestClip {\n")
rs.append("    pub expected_label: &'static str,\n")
rs.append("    pub source_file: &'static str,\n")
rs.append("    pub audio: &'static [f32],\n")
rs.append("}\n\n")

for i, (_label, audio) in enumerate(ordered, 1):
    audio_vals = ", ".join(f"{float(v):.8f}" for v in audio)
    rs.append(f"pub const CLIP_{i}: &[f32] = &[{audio_vals}];\n\n")

rs.append("pub const TEST_CLIPS: [TestClip; 4] = [\n")
for i, (label, _audio) in enumerate(ordered, 1):
    rs.append("    TestClip {\n")
    rs.append(f"        expected_label: \"{label}\",\n")
    rs.append(f"        source_file: \"dataset/testing/{label}/sample_{i}.wav\",\n")
    rs.append(f"        audio: CLIP_{i},\n")
    rs.append("    },\n")
rs.append("];\n")

OUT_AUDIO_RS.write_text("".join(rs), encoding="utf-8")
print("Wrote", OUT_AUDIO_RS, "clips=", len(ordered), "samples_per_clip=", len(ordered[0][1]))